<a href="https://colab.research.google.com/github/cathrineq/python-ai-Tarasova-Kate/blob/main/notebooks/week2b_read_csv.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 📊 Week 2: Data Analysis — Чтение и проверка данных

**Цель**: Научиться читать CSV-файлы из репозитория GitHub в Google Colab и выполнять базовую проверку данных с помощью pandas.

**Данные:**
- [`currency_rates.csv`](https://github.com/cathrineq/python-ai-Tarasova-Kate/blob/main/data/currency_rates.csv) — курсы обмена валют по времени (Wikidata P2284, P585, P580)
- [`countries_currencies.csv`](https://github.com/cathrineq/python-ai-Tarasova-Kate/blob/main/data/countries_currencies.csv) — страны и их официальные валюты (Wikidata P38, P1082, P2046)

**Что мы делаем:**
1. Клонируем репозиторий GitHub в Colab
2. Читаем CSV-файлы в pandas DataFrame
3. Очищаем и переименовываем столбцы
4. Смотрим структуру данных и делаем быструю валидацию

## 🐱 [1] Клонируем репозиторий курса в Colab

In [6]:
# 🐱 Шаг 1. Клонируем репозиторий курса в Colab

import os

repo = "python-ai-Tarasova-Kate"  # ← изменено: имя вашего репозитория
repo_path = f"/content/{repo}"  # абсолютный путь — не зависит от cwd

if not os.path.exists(repo_path):          # всегда проверяет /content/python-ai-Tarasova-Kate
    !git clone -q https://github.com/cathrineq/python-ai-Tarasova-Kate.git  # ← изменено: ваш URL репозитория

if os.getcwd() != repo_path:               # точное сравнение, не endswith
    %cd {repo_path}

print("✅ Репозиторий готов, теперь мы работаем внутри папки", repo)

✅ Репозиторий готов, теперь мы работаем внутри папки python-ai-Tarasova-Kate


## 📥 [2A] Простое чтение CSV-файлов в pandas

Сначала просто прочитаем оба CSV-файла в объекты `DataFrame`, без каких‑либо изменений.

После этого мы узнаем, сколько строк загружено в каждый датасет.

In [12]:
# 🐱 Шаг 2A. Чтение CSV-файлов в pandas

import pandas as pd
import os

print("🔍 Проверка доступных файлов в data/:")
if os.path.exists("data"):
    print(os.listdir("data"))
else:
    print("⚠️ Папка data/ не найдена!")

# Инициализируем переменные
df_rates = None
df_countries = None
df_currency = None

# Пробуем загрузить новые файлы
if os.path.exists("data/currency_rates.csv"):
    df_rates = pd.read_csv("data/currency_rates.csv")
    print("✅ Загружено df_rates:", len(df_rates), "строк")

if os.path.exists("data/countries_currencies.csv"):
    df_countries = pd.read_csv("data/countries_currencies.csv")
    print("✅ Загружено df_countries:", len(df_countries), "строк")

# Если новые файлы не найдены — используем исходный
if df_rates is None and df_countries is None:
    df_currency = pd.read_csv("data/currency.csv")
    print("✅ Загружено df_currency:", len(df_currency), "строк")

🔍 Проверка доступных файлов в data/:
['examples', 'currency.sparql', 'currency.csv', '.gitkeep', 'README.md']
✅ Загружено df_currency: 2348 строк


## 🧹 [2B] Очистка и переименование столбцов

В CSV-файлах из Wikidata есть **технические столбцы**, которые нужно обработать:

### Файл `currency_rates.csv`:
- Столбец `currency` с URL — **удаляем** (оставляем только `currencyLabel`)
- `currencyLabel → currency`, `unitLabel → unit` (убираем постфикс `Label`)
- Числовые: `price`, `year`, `startYear` → приводим к `float`/`int`

### Файл `countries_currencies.csv`:
- Столбцы `country`, `currency` с URL — **удаляем** (оставляем `*Label`)
- `countryLabel → country`, `currencyLabel → currency`
- Числовые: `population`, `area` → приводим к `int`/`float`

> ⚠️ **Важно:** URL Wikidata можно сохранить для отладки, но для анализа удобнее работать с читаемыми названиями.

In [13]:
# 🧹 Шаг 2B. Очистка и переименование столбцов

import pandas as pd
import os

# ========== ПРОВЕРКА: какие файлы доступны? ==========
print("🔍 Проверка доступных файлов...")

file_rates = "data/currency_rates.csv"
file_countries = "data/countries_currencies.csv"
file_currency = "data/currency.csv"

df_rates = None
df_countries = None
df_currency = None

# Пробуем загрузить новые файлы
if os.path.exists(file_rates):
    df_rates = pd.read_csv(file_rates)
    print("✅ Найден:", file_rates)
else:
    print("⚠️ Не найден:", file_rates)

if os.path.exists(file_countries):
    df_countries = pd.read_csv(file_countries)
    print("✅ Найден:", file_countries)
else:
    print("⚠️ Не найден:", file_countries)

# Если новые файлы не найдены — используем исходный currency.csv
if df_rates is None and df_countries is None:
    if os.path.exists(file_currency):
        df_currency = pd.read_csv(file_currency)
        print("✅ Используем исходный файл:", file_currency)
    else:
        print("❌ Ошибка: ни один CSV-файл не найден!")

# ========== ОЧИСТКА df_rates (если загружен) ==========
if df_rates is not None:
    print("\n🧹 Очистка df_rates (курсы валют)...")

    # Удаляем столбцы с URL
    df_rates = df_rates.drop(columns=["currency"], errors="ignore")

    # Переименовываем Label-столбцы
    df_rates = df_rates.rename(columns={
        "currencyLabel": "currency",
        "unitLabel": "unit"
    }, errors="ignore")

    # Числовые столбцы
    df_rates["price"] = pd.to_numeric(df_rates["price"], errors="coerce").fillna(0).astype(float)
    df_rates["year"] = pd.to_numeric(df_rates["year"], errors="coerce").fillna(0).astype(int)
    if "startYear" in df_rates.columns:
        df_rates["startYear"] = pd.to_numeric(df_rates["startYear"], errors="coerce").fillna(0).astype(int)

    print("✅ df_rates очищен")

# ========== ОЧИСТКА df_countries (если загружен) ==========
if df_countries is not None:
    print("\n🧹 Очистка df_countries (страны и валюты)...")

    # Удаляем столбцы с URL
    df_countries = df_countries.drop(columns=["country", "currency"], errors="ignore")

    # Переименовываем Label-столбцы
    df_countries = df_countries.rename(columns={
        "countryLabel": "country",
        "currencyLabel": "currency"
    }, errors="ignore")

    # Числовые столбцы
    df_countries["population"] = pd.to_numeric(df_countries["population"], errors="coerce").fillna(0).astype(int)
    df_countries["area"] = pd.to_numeric(df_countries["area"], errors="coerce").fillna(0).astype(float)

    print("✅ df_countries очищен")

# ========== ОЧИСТКА df_currency (исходный файл, если загружен) ==========
if df_currency is not None:
    print("\n🧹 Очистка df_currency (исходный файл)...")

    # Удаляем столбец с URL
    df_currency = df_currency.drop(columns=["currency"], errors="ignore")

    # Переименовываем Label-столбцы
    df_currency = df_currency.rename(columns={
        "currencyLabel": "currency",
        "unitLabel": "unit"
    }, errors="ignore")

    # Числовые столбцы
    df_currency["price"] = pd.to_numeric(df_currency["price"], errors="coerce").fillna(0).astype(float)
    df_currency["year"] = pd.to_numeric(df_currency["year"], errors="coerce").fillna(0).astype(int)

    print("✅ df_currency очищен")

print("\n✅ Очистка завершена")

🔍 Проверка доступных файлов...
⚠️ Не найден: data/currency_rates.csv
⚠️ Не найден: data/countries_currencies.csv
✅ Используем исходный файл: data/currency.csv

🧹 Очистка df_currency (исходный файл)...
✅ df_currency очищен

✅ Очистка завершена


## 🔍 [3] Обзор данных: структура и первые строки

Сделаем короткий обзор DataFrame `df_currency`:

- посмотрим размер таблицы (`shape`);
- выведем список столбцов;
- посмотрим первые несколько строк;
- дополнительно посчитаем базовую статистику по числовым столбцам (`price`, `year`).

Для удобства напишем маленькую функцию `show_info(df, name)`, чтобы быстро получать сводку по данным.

**Столбцы в нашем датасете:**
- `currency` — название валюты (например, «иракский динар»)
- `shortName` — короткий код/аббревиатура
- `price` — цена валюты в евро (числовой)
- `year` — год данных (числовой)
- `unit` — единица измерения (например, «евро»)
- `unitSymbol` — символ единицы (например, «د.ع»)

In [15]:
def show_info(df, name, n=5):
    """Краткий обзор DataFrame: имя, размер, столбцы, первые строки и статистика."""

    # 🔒 Проверка: DataFrame не должен быть None
    if df is None:
        print(f"⚠️ {name}: данные не загружены (None)")
        return

    print(f"\n📊 {name}")
    print("Размер:", df.shape)
    print("Столбцы:", ", ".join(df.columns))
    print("\nПервые строки:")
    print(df.head(n))

    numeric_cols = df.select_dtypes(include='number').columns
    if len(numeric_cols) > 0:
        print(f"\n📈 Статистика по числовым столбцам:")
        print(df[numeric_cols].describe())

# 🔍 Шаг 3. Обзор данных (с проверкой на None)
print("📋 Доступные данные для анализа:")

if df_rates is not None:
    show_info(df_rates, "Курсы валют (df_rates)")
else:
    print("⏭️ df_rates: файл не найден, пропускаем")

if df_countries is not None:
    show_info(df_countries, "Страны и валюты (df_countries)")
else:
    print("⏭️ df_countries: файл не найден, пропускаем")

# Если новые файлы не загружены — показываем исходный df_currency
if df_currency is not None:
    show_info(df_currency, "Валюты (df_currency) — исходный файл")
elif df_rates is None and df_countries is None:
    print("\n❌ Ошибка: ни один DataFrame не загружен!")

📋 Доступные данные для анализа:
⏭️ df_rates: файл не найден, пропускаем
⏭️ df_countries: файл не найден, пропускаем

📊 Валюты (df_currency) — исходный файл
Размер: (2348, 6)
Столбцы: currency, shortName, price, year, unit, unitSymbol

Первые строки:
           currency shortName    price  year        unit unitSymbol
0    иракский динар       NaN  0.00071  2020        евро        د.ع
1    иракский динар       NaN  0.00071  2020        евро         ID
2  кувейтский динар       NaN  3.28190  2016  доллар США        د.ك
3  кувейтский динар       NaN  3.00000  2016  доллар США        د.ك
4  кувейтский динар       NaN  3.28190  2016  доллар США         KD

📈 Статистика по числовым столбцам:
              price         year
count  2.348000e+03  2348.000000
mean   5.549886e+05  1013.577087
std    2.094162e+07  1010.418892
min    0.000000e+00     0.000000
25%    0.000000e+00     0.000000
50%    5.000000e-01  1855.500000
75%    2.000000e+00  2025.000000
max    1.000000e+09  2026.000000


## ✅ [4] Быстрая проверка и валидация данных

Здесь мы посмотрим:

- сколько **уникальных** валют, единиц измерения и символов есть в данных;
- **какие единицы измерения встречаются чаще всего** (Топ‑5 по числу строк);
- **какие валюты представлены в датасете** (список уникальных названий);
- **диапазон цен и лет**: минимальная/максимальная цена валюты в евро, период данных.

Функция `value_counts()`:
- считает, сколько раз каждое значение встречается в столбце;
- сортирует результаты по убыванию.

Метод `.head()` берёт первые N строк, поэтому
`df_currency["unit"].value_counts().head()` даёт **Топ‑5 единиц измерения по числу записей**.

> 💡 **Подсказка:** если в данных есть дубликаты (одна и та же валюта с разными символами), `value_counts()` поможет их быстро обнаружить.

In [16]:
# ✅ Шаг 4. Быстрая проверка и валидация данных

print("🔍 Быстрая проверка данных")

# ========== ФАЙЛ A: Курсы валют (если загружен) ==========
if df_rates is not None:
    print("\n📊 Датасет: Курсы валют (df_rates)")
    print("Уникальных валют:", df_rates["currency"].nunique())
    print("Уникальных единиц измерения:", df_rates["unit"].nunique())
    print(f"Диапазон цен: {df_rates['price'].min():.6f} — {df_rates['price'].max():.6f} евро")
    print(f"Диапазон лет: {df_rates['year'].min()} — {df_rates['year'].max()}")

    print("\nТоп-5 единиц измерения:")
    print(df_rates["unit"].value_counts().head())
else:
    print("\n⏭️ df_rates: пропускаем (файл не найден)")

# ========== ФАЙЛ B: Страны и валюты (если загружен) ==========
if df_countries is not None:
    print("\n📊 Датасет: Страны и валюты (df_countries)")
    print("Уникальных стран:", df_countries["country"].nunique())
    print("Уникальных валют:", df_countries["currency"].nunique())
    print(f"Население: {df_countries['population'].min():,} — {df_countries['population'].max():,}")
    print(f"Площадь: {df_countries['area'].min():,.0f} — {df_countries['area'].max():,.0f} км²")

    print("\nТоп-5 стран по числу записей:")
    print(df_countries["country"].value_counts().head())

    print("\nТоп-5 валют по числу стран:")
    print(df_countries["currency"].value_counts().head())
else:
    print("\n⏭️ df_countries: пропускаем (файл не найден)")

# ========== ИСХОДНЫЙ ФАЙЛ: currency.csv (если загружен) ==========
if df_currency is not None:
    print("\n📊 Датасет: Валюты (df_currency) — исходный файл")
    print("Уникальных валют:", df_currency["currency"].nunique())
    print("Уникальных единиц измерения:", df_currency["unit"].nunique())
    print(f"Диапазон цен: {df_currency['price'].min():.6f} — {df_currency['price'].max():.6f} евро")
    print(f"Диапазон лет: {df_currency['year'].min()} — {df_currency['year'].max()}")

    print("\nТоп-5 единиц измерения:")
    print(df_currency["unit"].value_counts().head())

    print("\n🔣 Символы единиц измерения:")
    print(df_currency["unitSymbol"].value_counts())
else:
    print("\n⏭️ df_currency: пропускаем (файл не найден)")

print("\n✅ Валидация завершена")

🔍 Быстрая проверка данных

⏭️ df_rates: пропускаем (файл не найден)

⏭️ df_countries: пропускаем (файл не найден)

📊 Датасет: Валюты (df_currency) — исходный файл
Уникальных валют: 1009
Уникальных единиц измерения: 129
Диапазон цен: 0.000000 — 1000000000.000000 евро
Диапазон лет: 0 — 2026

Топ-5 единиц измерения:
unit
доллар США         361
евро               270
фунт стерлингов     70
датская крона       31
шведская крона      30
Name: count, dtype: int64

🔣 Символы единиц измерения:
unitSymbol
€      855
kr      54
$       41
¥       34
₽       30
      ... 
T$       1
R₣       1
FRw      1
RWF      1
RF       1
Name: count, Length: 206, dtype: int64

✅ Валидация завершена


## 📝 Summary

**Что мы сделали в этом ноутбуке (Week 2):**

- ✅ Клонировали репозиторий GitHub в Colab
- ✅ Прочитали 2 CSV-файла из `data/examples/`
- ✅ Удалили URL Wikidata и переименовали столбцы (`*Label → короткие имена`)
- ✅ Проверили структуру данных (размер, столбцы, первые строки)
- ✅ Выполнили быструю валидацию:
  - количество уникальных фильмов, стран, жанров
  - диапазоны значений
  - топ стран и жанров по числу записей
  - типы оценок и результатов

Теперь у нас есть **аккуратные, проверенные таблицы**, с которыми удобно работать дальше.

В отдельном ноутбуке для следующей недели мы будем использовать **те же данные** для:
- более сложного анализа (группировки, фильтрация),
- и построения визуализаций (графики и диаграммы). 🎨